# Evaluator-Optimizer: Iterative Refinement Through Feedback Loops

**What you'll learn:**
- The Generator + Evaluator feedback loop architecture
- How to design effective evaluators with clear criteria
- Memory accumulation across iterations
- When iterative refinement justifies the extra cost
- Setting guardrails (max iterations, diminishing returns)

**Position on the spectrum:** The most quality-focused workflow — trades cost and latency for demonstrably better outputs through iteration.

> *"One LLM call generates a response while another provides evaluation and feedback in a loop."*

## How It Works

![Evaluator-Optimizer workflow — Generator and Evaluator in a feedback loop until the solution is accepted](assets/evaluator_optimizer.webp)

**The feedback loop:**

```
Task → Generator → Solution → Evaluator → PASS? → Output
                      ↑                       │
                      └── Feedback ←── NO ────┘
```

**Two roles:**
- **Generator:** Produces solutions, incorporating feedback from previous attempts
- **Evaluator:** Assesses solutions against clear criteria, provides actionable feedback

**Key mechanics:**
- Memory accumulates: the generator sees ALL previous attempts + their feedback
- The evaluator returns either `PASS` (done) or `NEEDS_IMPROVEMENT` + specific feedback
- A maximum iteration limit prevents infinite loops
- Diminishing returns typically set in after 2-3 rounds

## When to Use Evaluator-Optimizer

✅ **The two signs of good fit:**
1. LLM responses can be **demonstrably improved** when feedback is provided
2. The LLM can provide **meaningful feedback** on its own outputs

✅ **Use when:**
- Clear, measurable evaluation criteria exist
- Iterative refinement adds demonstrable value over single-pass
- Quality matters more than speed/cost
- You can define what "good enough" looks like

❌ **Don't use when:**
- No clear evaluation criteria (evaluator can't provide useful feedback)
- Single-pass quality is already sufficient
- Latency is critical (each iteration adds a full roundtrip)
- The task doesn't benefit from iteration (e.g., simple factual lookups)

In [ ]:
import sys
sys.path.append(".")
from util import llm_call, extract_xml

In [ ]:
def generate(prompt: str, task: str, context: str = "") -> tuple[str, str]:
    """Generate a solution, optionally incorporating feedback from previous attempts."""
    full_prompt = f"{prompt}\n{context}\nTask: {task}" if context else f"{prompt}\nTask: {task}"
    response = llm_call(full_prompt)
    thoughts = extract_xml(response, "thoughts")
    result = extract_xml(response, "response")
    
    print("── GENERATOR ──")
    print(f"  Thinking: {thoughts[:200]}...")
    print(f"  Output: {result[:200]}...")
    return thoughts, result


def evaluate(prompt: str, content: str, task: str) -> tuple[str, str]:
    """Evaluate a solution against criteria. Returns verdict + feedback."""
    full_prompt = f"{prompt}\nOriginal task: {task}\nContent to evaluate: {content}"
    response = llm_call(full_prompt)
    evaluation = extract_xml(response, "evaluation")
    feedback = extract_xml(response, "feedback")
    
    print("── EVALUATOR ──")
    print(f"  Verdict: {evaluation}")
    print(f"  Feedback: {feedback[:200]}...")
    return evaluation, feedback


def optimization_loop(task: str, evaluator_prompt: str, generator_prompt: str, max_iterations: int = 3) -> tuple[str, list]:
    """Run the generate → evaluate → refine loop until PASS or max iterations."""
    memory = []
    chain_of_thought = []
    
    print(f"{'═' * 60}")
    print(f"  ITERATION 1 (initial generation)")
    print(f"{'═' * 60}")
    
    thoughts, result = generate(generator_prompt, task)
    memory.append(result)
    chain_of_thought.append({"thoughts": thoughts, "result": result, "iteration": 1})
    
    for iteration in range(2, max_iterations + 1):
        evaluation, feedback = evaluate(evaluator_prompt, result, task)
        
        if evaluation.strip().upper() == "PASS":
            print(f"\n✓ PASSED on iteration {iteration - 1}")
            return result, chain_of_thought
        
        print(f"\n{'═' * 60}")
        print(f"  ITERATION {iteration} (refining based on feedback)")
        print(f"{'═' * 60}")
        
        # Build context from all previous attempts + feedback
        context = "\n".join([
            "Previous attempts:",
            *[f"- Attempt {i+1}: {m[:100]}..." for i, m in enumerate(memory)],
            f"\nLatest feedback: {feedback}",
            "\nImprove based on this feedback while keeping what already works."
        ])
        
        thoughts, result = generate(generator_prompt, task, context)
        memory.append(result)
        chain_of_thought.append({"thoughts": thoughts, "result": result, "iteration": iteration})
    
    print(f"\n⚠ Max iterations ({max_iterations}) reached")
    return result, chain_of_thought

## Example 1: Code Quality Improvement (MinStack)

A task with clear evaluation criteria: implement a data structure that must be correct, efficient, AND well-documented. The evaluator catches gaps the generator misses on first pass.

In [ ]:
evaluator_prompt = """Evaluate this code implementation for:
1. Correctness — does it solve the task with no bugs?
2. Time complexity — do all operations meet the O(1) requirement?
3. Code quality — type hints, docstrings, error handling, clean style?

Be strict: only output "PASS" if ALL criteria are fully met with no remaining suggestions.

<evaluation>PASS or NEEDS_IMPROVEMENT</evaluation>
<feedback>
Specific issues to fix, with examples of what good looks like.
</feedback>"""

generator_prompt = """Complete the coding task below. If there is feedback from previous attempts,
reflect on it and improve your solution.

Output your thinking and solution:
<thoughts>
Your understanding of the task and how you'll improve based on any feedback.
</thoughts>
<response>
Your complete code implementation.
</response>"""

task = """Implement a MinStack class with:
1. push(x) — push element onto stack
2. pop() — remove top element
3. getMin() — retrieve minimum element
All operations must be O(1) time complexity."""

result, history = optimization_loop(task, evaluator_prompt, generator_prompt, max_iterations=4)

print(f"\n{'═' * 60}")
print("  FINAL RESULT")
print(f"{'═' * 60}")
print(result)
print(f"\n  Total iterations: {len(history)}")

## Designing Effective Evaluators

The evaluator is the quality function that drives improvement. A weak evaluator = weak results.

### What makes a good evaluator:

| Property | Bad Example | Good Example |
|----------|-------------|--------------|
| **Specific criteria** | "Is this code good?" | "Does it handle empty stack? Are all ops O(1)?" |
| **Actionable feedback** | "Needs improvement" | "Missing type hints on push() — add `x: int`" |
| **Binary verdict** | "7/10" | "PASS" or "NEEDS_IMPROVEMENT" |
| **Non-conflicting** | "Be concise" + "Add more detail" | "Be concise in comments, detailed in docstrings" |

### ACI Principle Applied: Clear Evaluation Interface

The evaluator prompt is a tool interface — the generator "uses" the feedback to improve. Apply ACI principles:
- **Poka-yoke:** Force structured output (PASS/NEEDS_IMPROVEMENT only, not free-text verdicts)
- **Document like onboarding:** Tell the evaluator exactly what to check
- **Format for LLM strengths:** Separate verdict from feedback (model commits to binary decision first)

## Example 2: Writing Quality Refinement

A non-code example showing the pattern's versatility. Clear criteria + iterative improvement works for any domain where quality is measurable.

In [ ]:
writing_evaluator = """Evaluate this technical explanation for:
1. Clarity — would a developer with 2 years experience understand it?
2. Accuracy — are all technical claims correct?
3. Conciseness — is there unnecessary repetition or filler?
4. Structure — does it flow logically from simple to complex?

Only PASS if all criteria are met. Be specific in feedback.

<evaluation>PASS or NEEDS_IMPROVEMENT</evaluation>
<feedback>
Specific issues and how to fix them.
</feedback>"""

writing_generator = """Write a clear technical explanation for the given topic.
Target audience: developers with 2 years experience.
Constraints: Under 200 words, no jargon without definition, include one concrete example.

If there is feedback from previous attempts, improve based on it.

<thoughts>
Your approach and how you're addressing any feedback.
</thoughts>
<response>
Your explanation here.
</response>"""

topic = """Task: Explain why database connection pooling improves application performance."""

result, history = optimization_loop(topic, writing_evaluator, writing_generator, max_iterations=3)

print(f"\n{'═' * 60}")
print("  FINAL EXPLANATION")
print(f"{'═' * 60}")
print(result)

## Pitfalls & Guardrails

| Pitfall | What happens | Mitigation |
|---------|-------------|------------|
| **No max iterations** | Infinite loop, unbounded cost | Always set a limit (3-4 rounds typical) |
| **Weak evaluator** | Passes bad output or gives vague feedback | Test evaluator independently with known-bad inputs |
| **Evaluator too strict** | Never passes, always finds something | Calibrate: "only suggest improvements that are HIGH impact" |
| **Generator ignores feedback** | Same mistakes repeated | Include previous attempts in context so model sees the pattern |
| **Overfitting to evaluator** | Generator games the evaluator's criteria | Use diverse evaluation criteria, not just one metric |
| **Diminishing returns** | Iteration 4+ makes negligible improvements | Track quality delta between iterations, stop when small |

### Combining with Other Patterns

The evaluator-optimizer can wrap ANY other pattern:
- **Chain output → evaluate → refine chain**: Improve a pipeline's final output
- **Orchestrator output → evaluate → re-orchestrate**: Improve task decomposition
- **Parallel outputs → evaluate best → refine winner**: Select and polish

## Key Takeaways

1. **Two roles, one loop:** Generator produces, Evaluator critiques, repeat until PASS
2. **Clear criteria are everything** — the evaluator is only as good as its rubric
3. **Memory enables learning** — generator sees all previous attempts + feedback
4. **Set max iterations** — diminishing returns after 2-3 rounds in practice
5. **Cost trade-off is explicit** — 2-4× the calls for demonstrably better output

---

**Next up:** [06_autonomous_agents.ipynb](06_autonomous_agents.ipynb) — From predefined workflows to fully autonomous agents that direct their own behavior.